In [ ]:
import numpy as np
import pandas as pd

def evaluate_performance(arch):
    """
    Evaluates the 4 performance metrics based on architectural decisions.
    Input `arch` is a dictionary with keys 'd1' to 'd8'.
    """

    # ---------------------------------------------------------
    # Fixed Operational Assumptions for GA Aircraft Class
    # ---------------------------------------------------------
    POWER_REQUIRED_KW = 200.0     # Estimated cruise power
    CRUISE_SPEED_MPH = 150.0      # Average cruise speed
    TOTAL_STORAGE_MASS_KG = 300.0 # Assumed total payload available for fuel + battery

    # Energy Densities
    FUEL_DENSITY_MJ_KG = 43.0     # Aviation fuel
    # Battery Energy Densities (Wh/kg) converted to MJ/kg (1 Wh = 0.0036 MJ)
    # D2 -> 0: NMC (250), 1: LFP (180), 2: Solid-state (400)
    BATTERY_DENSITY_MJ_KG = {
        0: 250 * 0.0036,
        1: 180 * 0.0036,
        2: 400 * 0.0036
    }

    # CO2 Emission Factor (Jet Fuel: ~9.57 kg CO2/gal -> ~3.11 kg CO2/kg fuel)
    EMISSION_FACTOR_KG_CO2_PER_KG_FUEL = 3.11

    # Baseline Motor Reliability
    R_MOTOR = 0.98

    # ---------------------------------------------------------
    # Architectural Decoding & Mass Allocation
    # ---------------------------------------------------------
    d1_topology = arch['d1']
    d2_battery = arch['d2']
    d3_motors = arch['d3']

    # Map D1 (Topology) to Propulsion Efficiency
    if d1_topology == 0 or d1_topology == 1:
        # Series or Parallel Hybrid
        eta_propulsion = 0.40
        fuel_mass_kg = 100.0
        battery_mass_kg = 200.0
    else:
        # Turboelectric (proxy for Turboprop efficiency)
        eta_propulsion = 0.35
        # Turboelectric heavily relies on fuel driving a generator
        fuel_mass_kg = 250.0
        battery_mass_kg = 50.0

    # Map D3 (Motor Count) to Integer
    motor_counts = {0: 1, 1: 2, 2: 4}
    n_motors = motor_counts[d3_motors]

    # ---------------------------------------------------------
    # Metric P1: Energy Efficiency (MJ consumed per mile)
    # ---------------------------------------------------------
    # Energy Consumption (MJ/hr) = (kW * 3600 sec/hr) / 1000 / efficiency
    energy_consumption_mj_hr = (POWER_REQUIRED_KW * 3.6) / eta_propulsion

    # Energy per mile = (MJ/hr) / (miles/hr)
    p1_efficiency_mj_mile = energy_consumption_mj_hr / CRUISE_SPEED_MPH

    # ---------------------------------------------------------
    # Metric P2: Operational Range (Miles)
    # ---------------------------------------------------------
    # Calculate stored energy
    fuel_energy_mj = fuel_mass_kg * FUEL_DENSITY_MJ_KG
    battery_energy_mj = battery_mass_kg * BATTERY_DENSITY_MJ_KG[d2_battery]
    total_energy_mj = fuel_energy_mj + battery_energy_mj

    p2_range_miles = total_energy_mj / p1_efficiency_mj_mile

    # ---------------------------------------------------------
    # Metric P3: CO2 Emissions per Mile (kg CO2 / mile)
    # ---------------------------------------------------------
    # Determine the fraction of energy coming from fuel combustion
    fuel_energy_fraction = fuel_energy_mj / total_energy_mj
    fuel_energy_per_mile_mj = p1_efficiency_mj_mile * fuel_energy_fraction

    fuel_mass_per_mile_kg = fuel_energy_per_mile_mj / FUEL_DENSITY_MJ_KG
    p3_emissions_co2_mile = fuel_mass_per_mile_kg * EMISSION_FACTOR_KG_CO2_PER_KG_FUEL

    # ---------------------------------------------------------
    # Metric P4: Propulsion System Reliability (%)
    # ---------------------------------------------------------
    # Reliability increases via parallel redundancy
    p4_reliability = 1.0 - (1.0 - R_MOTOR) ** n_motors

    return {
        'P1_Efficiency_MJ_per_mile': round(p1_efficiency_mj_mile, 2),
        'P2_Range_miles': round(p2_range_miles, 1),
        'P3_CO2_per_mile': round(p3_emissions_co2_mile, 3),
        'P4_Reliability_pct': round(p4_reliability * 100, 4)
    }

# ---------------------------------------------------------
# Testing with Reference Architectures
# ---------------------------------------------------------
# Reference Architecture 1: Cirrus SR22 Hybrid Concept (0 - 2 - 0 - ... )
arch_1 = {'d1': 0, 'd2': 2, 'd3': 0}

# Reference Architecture 2: Diamond DA62 Hybrid Retrofit (1 - 0 - 1 - ... )
arch_2 = {'d1': 1, 'd2': 0, 'd3': 1}

# Reference Architecture 3: NASA X-57 e-STOL (2 - 1 - 2 - ... )
arch_3 = {'d1': 2, 'd2': 1, 'd3': 2}

architectures = [arch_1, arch_2, arch_3]
results = []

for i, arch in enumerate(architectures, 1):
    metrics = evaluate_performance(arch)
    metrics['Architecture'] = f"Ref Arch {i}"
    results.append(metrics)

df_results = pd.DataFrame(results).set_index('Architecture')
print(df_results)

              P1_Efficiency_MJ_per_mile  P2_Range_miles  P3_CO2_per_mile  \
Architecture                                                               
Ref Arch 1                        12.00           382.3            0.813   
Ref Arch 2                        12.00           373.3            0.833   
Ref Arch 3                        13.71           786.2            0.989   

              P4_Reliability_pct  
Architecture                      
Ref Arch 1                 98.00  
Ref Arch 2                 99.96  
Ref Arch 3                100.00  
